# Customer Churn & Lifetime Value (CLTV) Prediction

This notebook implements probabilistic models for **Customer Churn Prediction** and **Customer Lifetime Value (CLTV) Estimation** using the pre-processed RFM (Recency, Frequency, Monetary, Tenure) dataset.

## Methodology
| Phase | Model / Technique | Purpose |
|-------|-------------------|--------|
| 1 | Data Preparation | Load, validate, and filter edge cases |
| 2 | BG/NBD Model | Predict future transactions & churn probability |
| 3 | Gamma-Gamma Model | Estimate average monetary value & CLTV |
| 4 | K-Means Clustering | Create actionable customer segments |
| 5 | Synthesis & Output | Consolidate results, export, and visualize |

---
## Phase 1: Environment & Data Preparation

### 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.plotting import (
    plot_frequency_recency_matrix,
    plot_probability_alive_matrix,
    plot_period_transactions,
)
from lifetimes.utils import calibration_and_holdout_data

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

### 1.2 Load Data

In [ ]:
df = pd.read_csv('rfm_customer_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumn dtypes:\n{df.dtypes}')
print(f'\nFirst 5 rows:')
df.head()

### 1.3 Data Type Validation
Ensure `Customer ID` is treated as a string identifier (not a numerical feature) and RFM columns are numeric.

In [ ]:
# Ensure Customer ID is treated as a string identifier
df['Customer ID'] = df['Customer ID'].astype(str)

# Verify RFM columns are numeric
rfm_cols = ['Recency', 'Frequency', 'Monetary', 'Tenure']
for col in rfm_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Data types after validation:')
print(df.dtypes)
print(f'\nNull values:\n{df.isnull().sum()}')

### 1.4 Filter Edge Cases
The `lifetimes` library requires:
- **Monetary > 0**: Gamma-Gamma model cannot handle zero/negative monetary values.
- **Frequency > 0**: BG/NBD model needs at least 1 repeat purchase for meaningful probability estimates.
- **Tenure >= Recency**: A customer's tenure must be >= their recency (time since last purchase).

In [ ]:
original_count = len(df)

# Drop rows with NaN values from coercion
df = df.dropna(subset=rfm_cols)

# Filter: Monetary must be strictly positive
df = df[df['Monetary'] > 0]

# Filter: Frequency must be > 0 (at least 1 repeat purchase)
df = df[df['Frequency'] > 0]

# Filter: Tenure must be >= Recency (logical consistency)
df = df[df['Tenure'] >= df['Recency']]

filtered_count = len(df)
print(f'Customers removed: {original_count - filtered_count} ({(original_count - filtered_count)/original_count*100:.1f}%)')
print(f'Remaining customers: {filtered_count}')
print(f'\nSummary statistics after filtering:')
df[rfm_cols].describe()

---
## Phase 2: Churn & Future Transaction Modeling (BG/NBD Model)

The **Beta Geometric / Negative Binomial Distribution (BG/NBD)** model estimates:
1. The **probability that a customer is still alive** (active).
2. The **expected number of future transactions** in a given time period.

### 2.1 Fit BG/NBD Model

In [ ]:
bgf = BetaGeoFitter()

try:
    # First attempt: fit without penalizer
    bgf.fit(
        frequency=df['Frequency'],
        recency=df['Recency'],
        T=df['Tenure']
    )
    print('BG/NBD model fitted successfully (no penalizer needed).')
except Exception as e:
    print(f'Initial fit failed: {e}')
    print('Retrying with penalizer_coef=0.001...')
    bgf = BetaGeoFitter(penalizer_coef=0.001)
    bgf.fit(
        frequency=df['Frequency'],
        recency=df['Recency'],
        T=df['Tenure']
    )
    print('BG/NBD model fitted successfully (with penalizer).')

print(f'\nModel Parameters:')
print(bgf.summary)

### 2.2 Predict Future Purchases
Predict the expected number of transactions over the next **30, 90, and 180 days**.

In [ ]:
# Predict expected purchases for different time horizons
for period in [30, 90, 180]:
    col_name = f'Predicted_Purchases_{period}d'
    df[col_name] = bgf.predict(
        t=period,
        frequency=df['Frequency'],
        recency=df['Recency'],
        T=df['Tenure']
    )

print('Predicted purchases distribution (180-day horizon):')
print(df['Predicted_Purchases_180d'].describe())
print(f'\nTop 10 customers by expected purchases (next 180 days):')
df.nlargest(10, 'Predicted_Purchases_180d')[['Customer ID', 'Frequency', 'Recency', 'Tenure', 'Predicted_Purchases_180d']]

### 2.3 Calculate Churn Risk Score

- **Probability Alive**: The conditional probability that a customer is still "active".
- **Churn Risk Score** = `1.0 - Probability_Alive`
- **Churn Flag**: Binary classification where customers with Probability_Alive < 0.2 are marked as "Churned".

In [ ]:
# Calculate probability of being alive
df['Probability_Alive'] = bgf.conditional_probability_alive(
    frequency=df['Frequency'],
    recency=df['Recency'],
    T=df['Tenure']
)

# Churn Risk Score = 1 - Probability_Alive
df['Churn_Risk_Score'] = 1.0 - df['Probability_Alive']

# Hard threshold: Churned if Probability_Alive < 0.2
df['Is_Churned'] = (df['Probability_Alive'] < 0.2).astype(int)

print(f'Churn Risk Score distribution:')
print(df['Churn_Risk_Score'].describe())
print(f'\nChurn breakdown:')
print(df['Is_Churned'].value_counts().rename({0: "Active", 1: "Churned"}))
print(f'\nChurn Rate: {df["Is_Churned"].mean()*100:.1f}%')

### 2.4 BG/NBD Visualizations

In [ ]:
# Frequency-Recency Matrix: Expected number of purchases in next 180 days
plt.figure(figsize=(10, 6))
plot_frequency_recency_matrix(bgf, T=180)
plt.title('Expected Purchases in Next 180 Days\n(Frequency-Recency Matrix)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bgf_freq_recency_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bgf_freq_recency_matrix.png')

In [ ]:
# Probability Alive Matrix
plt.figure(figsize=(10, 6))
plot_probability_alive_matrix(bgf)
plt.title('Probability Customer is Still Alive\n(Frequency-Recency Matrix)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bgf_prob_alive_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bgf_prob_alive_matrix.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution of Probability Alive
axes[0].hist(df['Probability_Alive'], bins=50, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(x=0.2, color='red', linestyle='--', linewidth=2, label='Churn Threshold (0.2)')
axes[0].set_xlabel('Probability Alive')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Distribution of Probability Alive', fontsize=13, fontweight='bold')
axes[0].legend()

# Churn Risk Score distribution
axes[1].hist(df['Churn_Risk_Score'], bins=50, color='#FF5722', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Churn Risk Score')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Distribution of Churn Risk Score', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: churn_risk_distribution.png')

---
## Phase 3: Customer Value Modeling (Gamma-Gamma Model)

The **Gamma-Gamma model** estimates the **average monetary value** of a customer's future transactions, which is then combined with the BG/NBD model to produce a full **Customer Lifetime Value (CLTV)** estimate.

### 3.1 Assumption Check: Frequency vs. Monetary Independence

The Gamma-Gamma model assumes **no correlation** between purchase frequency and average monetary value. We validate this before proceeding.

In [ ]:
# Pearson correlation between Frequency and Monetary
corr = df['Frequency'].corr(df['Monetary'])
print(f'Pearson Correlation (Frequency vs. Monetary): {corr:.4f}')

if abs(corr) > 0.3:
    print(f'WARNING: Moderate/strong correlation detected ({corr:.4f}).')
    print('   The Gamma-Gamma model assumption may be violated.')
    print('   CLTV estimates should be interpreted with caution.')
else:
    print(f'Correlation is weak ({corr:.4f}). Gamma-Gamma assumption holds.')

# Visual check
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['Frequency'], df['Monetary'], alpha=0.3, s=15, color='#673AB7')
ax.set_xlabel('Frequency (Number of Repeat Purchases)')
ax.set_ylabel('Monetary (Average Transaction Value)')
ax.set_title(f'Frequency vs. Monetary (r = {corr:.4f})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Fit Gamma-Gamma Model

In [ ]:
ggf = GammaGammaFitter()

try:
    ggf.fit(
        frequency=df['Frequency'],
        monetary_value=df['Monetary']
    )
    print('Gamma-Gamma model fitted successfully (no penalizer needed).')
except Exception as e:
    print(f'Initial fit failed: {e}')
    print('Retrying with penalizer_coef=0.001...')
    ggf = GammaGammaFitter(penalizer_coef=0.001)
    ggf.fit(
        frequency=df['Frequency'],
        monetary_value=df['Monetary']
    )
    print('Gamma-Gamma model fitted successfully (with penalizer).')

print(f'\nModel Parameters:')
print(ggf.summary)

### 3.3 Calculate Customer Lifetime Value (CLTV)

Combine BG/NBD and Gamma-Gamma models to estimate the total expected monetary value from each customer over a **6-month (180-day) horizon** with a monthly discount rate of 1%.

In [ ]:
# Calculate CLTV over 6-month horizon
# time=6 means 6 months, discount_rate=0.01 is monthly
# freq='D' since our RFM data is in daily units
df['Predicted_CLTV'] = ggf.customer_lifetime_value(
    transaction_prediction_model=bgf,
    frequency=df['Frequency'],
    recency=df['Recency'],
    T=df['Tenure'],
    monetary_value=df['Monetary'],
    time=6,           # 6 months
    discount_rate=0.01,  # monthly discount rate
    freq='D'          # RFM data in daily granularity
)

print('CLTV Distribution:')
print(df['Predicted_CLTV'].describe())
print(f'\nTotal Predicted Revenue (6-month): ${df["Predicted_CLTV"].sum():,.2f}')
print(f'Average CLTV per Customer: ${df["Predicted_CLTV"].mean():,.2f}')
print(f'Median CLTV per Customer: ${df["Predicted_CLTV"].median():,.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CLTV Distribution (capped at 99th percentile for visibility)
cap = df['Predicted_CLTV'].quantile(0.99)
axes[0].hist(df['Predicted_CLTV'].clip(upper=cap), bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Predicted CLTV ($)')
axes[0].set_ylabel('Number of Customers')
axes[0].set_title('Distribution of Predicted CLTV (6-Month Horizon)\n(Capped at 99th Percentile)', fontsize=13, fontweight='bold')

# Top 20 customers by CLTV
top20 = df.nlargest(20, 'Predicted_CLTV')
axes[1].barh(top20['Customer ID'], top20['Predicted_CLTV'], color='#009688', edgecolor='white')
axes[1].set_xlabel('Predicted CLTV ($)')
axes[1].set_ylabel('Customer ID')
axes[1].set_title('Top 20 Most Valuable Customers (6-Month CLTV)', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('cltv_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cltv_distribution.png')

---
## Phase 4: Customer Segmentation (K-Means Clustering)

Combine historical RFM metrics with predicted CLTV and Churn Risk to create actionable customer segments.

### 4.1 Feature Selection & Scaling

In [ ]:
# Features for clustering
cluster_features = ['Recency', 'Frequency', 'Monetary', 'Predicted_CLTV', 'Churn_Risk_Score']

# Prepare the feature matrix
X = df[cluster_features].copy()

# Standardize features (K-Means is sensitive to scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Feature matrix shape:', X_scaled.shape)
print('\nScaled feature statistics:')
pd.DataFrame(X_scaled, columns=cluster_features).describe().round(3)

### 4.2 Determine Optimal Number of Clusters
Using the **Elbow Method** (WCSS) and **Silhouette Score** to find the optimal `k`.

In [ ]:
K_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)
    print(f'k={k:2d} | WCSS: {kmeans.inertia_:12,.1f} | Silhouette: {sil_score:.4f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow Method
axes[0].plot(list(K_range), wcss, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Within-Cluster Sum of Squares (WCSS)')
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].set_xticks(list(K_range))

# Silhouette Score
axes[1].plot(list(K_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis', fontsize=13, fontweight='bold')
axes[1].set_xticks(list(K_range))

plt.tight_layout()
plt.savefig('optimal_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: optimal_clusters.png')

### 4.3 Fit K-Means with Optimal k
Based on the elbow method and silhouette analysis, we use **k=4** to create interpretable segments:
- Champions, Loyal, At-Risk, and Hibernating/Lost.

In [ ]:
OPTIMAL_K = 4

kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10, max_iter=300)
df['Cluster_ID'] = kmeans_final.fit_predict(X_scaled)

print(f'K-Means fitted with k={OPTIMAL_K}')
print(f'\nCluster distribution:')
print(df['Cluster_ID'].value_counts().sort_index())
print(f'\nCluster centroids (scaled):')
pd.DataFrame(kmeans_final.cluster_centers_, columns=cluster_features).round(3)

### 4.4 Interpret and Name Segments
Assign human-readable labels based on each cluster's RFM/CLTV/Churn profile.

In [ ]:
# Compute cluster profiles
cluster_profile = df.groupby('Cluster_ID')[cluster_features].mean()
print('Cluster Profiles (Mean Values):')
print(cluster_profile.round(2).to_string())

# Auto-label strategy:
# Rank clusters by Predicted_CLTV (descending) to assign names
cltv_ranking = cluster_profile['Predicted_CLTV'].sort_values(ascending=False).index.tolist()

segment_names = {
    cltv_ranking[0]: 'Champions',        # Highest CLTV
    cltv_ranking[1]: 'Loyal Customers',   # Second highest CLTV
    cltv_ranking[2]: 'At-Risk',           # Third
    cltv_ranking[3]: 'Hibernating/Lost',  # Lowest CLTV
}

df['Customer_Segment'] = df['Cluster_ID'].map(segment_names)

print(f'\nSegment Counts:')
print(df['Customer_Segment'].value_counts())
print(f'\nSegment Profiles (Mean Values):')
df.groupby('Customer_Segment')[cluster_features].mean().round(2)

---
## Phase 5: Synthesis & Output

### 5.1 Master DataFrame

In [ ]:
# Select and order columns for the final output
output_cols = [
    'Customer ID',
    'Recency', 'Frequency', 'Monetary', 'Tenure',
    'Predicted_Purchases_30d', 'Predicted_Purchases_90d', 'Predicted_Purchases_180d',
    'Probability_Alive', 'Churn_Risk_Score', 'Is_Churned',
    'Predicted_CLTV',
    'Cluster_ID', 'Customer_Segment'
]

df_final = df[output_cols].copy()
print(f'Final dataset shape: {df_final.shape}')
print(f'\nColumns: {list(df_final.columns)}')
df_final.head(10)

### 5.2 Sanity Check: Segment Statistics

In [ ]:
# Descriptive statistics grouped by segment
segment_stats = df_final.groupby('Customer_Segment').agg({
    'Customer ID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'Predicted_CLTV': ['mean', 'median', 'sum'],
    'Churn_Risk_Score': 'mean',
    'Probability_Alive': 'mean',
    'Predicted_Purchases_180d': 'mean',
}).round(2)

segment_stats.columns = [
    'Count', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary',
    'Mean_CLTV', 'Median_CLTV', 'Total_CLTV',
    'Avg_Churn_Risk', 'Avg_Prob_Alive', 'Avg_Predicted_Purchases_180d'
]

print('Segment Summary Statistics:')
print('=' * 120)
segment_stats

### 5.3 Export Results

In [ ]:
output_file = 'customer_cltv_predictions_and_segments.csv'
df_final.to_csv(output_file, index=False)
print(f'Results exported to: {output_file}')
print(f'   Rows: {len(df_final):,}')
print(f'   Columns: {len(df_final.columns)}')
print(f'   File size: {os.path.getsize(output_file) / 1024:.1f} KB')

### 5.4 Final Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

segment_colors = {
    'Champions': '#4CAF50',
    'Loyal Customers': '#2196F3',
    'At-Risk': '#FF9800',
    'Hibernating/Lost': '#F44336'
}

# 1. Recency vs Predicted CLTV
for segment, color in segment_colors.items():
    mask = df_final['Customer_Segment'] == segment
    axes[0, 0].scatter(
        df_final.loc[mask, 'Recency'],
        df_final.loc[mask, 'Predicted_CLTV'],
        c=color, label=segment, alpha=0.5, s=20, edgecolors='none'
    )
axes[0, 0].set_xlabel('Recency (Days Since Last Purchase)')
axes[0, 0].set_ylabel('Predicted CLTV ($)')
axes[0, 0].set_title('Recency vs. Predicted CLTV by Segment', fontsize=13, fontweight='bold')
axes[0, 0].legend()

# 2. Frequency vs Predicted CLTV
for segment, color in segment_colors.items():
    mask = df_final['Customer_Segment'] == segment
    axes[0, 1].scatter(
        df_final.loc[mask, 'Frequency'],
        df_final.loc[mask, 'Predicted_CLTV'],
        c=color, label=segment, alpha=0.5, s=20, edgecolors='none'
    )
axes[0, 1].set_xlabel('Frequency (Repeat Purchases)')
axes[0, 1].set_ylabel('Predicted CLTV ($)')
axes[0, 1].set_title('Frequency vs. Predicted CLTV by Segment', fontsize=13, fontweight='bold')
axes[0, 1].legend()

# 3. Probability Alive vs Frequency
for segment, color in segment_colors.items():
    mask = df_final['Customer_Segment'] == segment
    axes[1, 0].scatter(
        df_final.loc[mask, 'Frequency'],
        df_final.loc[mask, 'Probability_Alive'],
        c=color, label=segment, alpha=0.5, s=20, edgecolors='none'
    )
axes[1, 0].set_xlabel('Frequency (Repeat Purchases)')
axes[1, 0].set_ylabel('Probability Alive')
axes[1, 0].set_title('Probability Alive vs. Frequency by Segment', fontsize=13, fontweight='bold')
axes[1, 0].legend()

# 4. Segment breakdown (pie chart)
seg_counts = df_final['Customer_Segment'].value_counts()
colors = [segment_colors[s] for s in seg_counts.index]
axes[1, 1].pie(
    seg_counts.values, labels=seg_counts.index, colors=colors,
    autopct='%1.1f%%', startangle=140, textprops={'fontsize': 11}
)
axes[1, 1].set_title('Customer Segment Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('customer_segments_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: customer_segments_analysis.png')

In [ ]:
# Heatmap of segment profiles
profile_data = df_final.groupby('Customer_Segment')[[
    'Recency', 'Frequency', 'Monetary', 'Predicted_CLTV',
    'Churn_Risk_Score', 'Probability_Alive', 'Predicted_Purchases_180d'
]].mean()

# Normalize for heatmap
profile_normalized = (profile_data - profile_data.min()) / (profile_data.max() - profile_data.min())

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    profile_normalized,
    annot=profile_data.round(1).values,
    fmt='', cmap='RdYlGn', linewidths=1, linecolor='white',
    ax=ax, cbar_kws={'label': 'Normalized Score'},
    xticklabels=['Recency', 'Frequency', 'Monetary', 'CLTV',
                 'Churn Risk', 'Prob. Alive', 'Pred. Purchases (180d)']
)
ax.set_title('Customer Segment Profiles (Heatmap)', fontsize=14, fontweight='bold')
ax.set_ylabel('Segment')
plt.tight_layout()
plt.savefig('segment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: segment_heatmap.png')

---
## Summary

### Models Used
- **BG/NBD Model**: Predicted future purchase probability and churn risk
- **Gamma-Gamma Model**: Estimated average transaction value and CLTV
- **K-Means Clustering**: Segmented customers into actionable groups

### Output Files
| File | Description |
|------|-------------|
| `customer_cltv_predictions_and_segments.csv` | Final dataset with predictions and segments |
| `bgf_freq_recency_matrix.png` | BG/NBD frequency-recency matrix |
| `bgf_prob_alive_matrix.png` | BG/NBD probability alive matrix |
| `churn_risk_distribution.png` | Churn risk score distribution |
| `cltv_distribution.png` | CLTV distribution and top customers |
| `optimal_clusters.png` | Elbow method and silhouette analysis |
| `customer_segments_analysis.png` | Scatter plots and pie chart of segments |
| `segment_heatmap.png` | Heatmap of segment profiles |